# MLflow for MLOps — Complete Hands-On Tutorial

## Using the Advertising Dataset

This notebook teaches the complete MLflow workflow step by step using:

- **TV**
- **Radio**
- **Newspaper**
- **Sales**

The ML problem is:

\[
X = [TV,\ Radio,\ Newspaper]
\]

\[
y = Sales
\]

Our goal is not only to train models. We want to understand how **MLflow supports the ML model lifecycle in MLOps**.

---

## What you will learn

By the end of this notebook, you should understand:

1. What MLflow is
2. Experiments and runs
3. Parameters, metrics, and artifacts
4. Manual experiment tracking
5. Comparing several ML models
6. Logging trained models
7. Logging plots and other artifacts
8. MLflow autologging
9. Train/test metrics and overfitting gaps
10. Model Registry
11. Model versions
12. Model aliases such as `champion`
13. Loading registered models
14. Model signatures
15. Serving a model as an API
16. MLflow architecture
17. Model lineage and reproducibility

---

# The Big Picture

\[
\boxed{
\text{Data}
\rightarrow
\text{Experimentation}
\rightarrow
\text{Training}
\rightarrow
\text{Evaluation}
\rightarrow
\text{Registry}
\rightarrow
\text{Deployment}
}
\]

MLflow mainly helps us manage the journey from experimentation to production model management.

# 1. Install MLflow

Run this only if MLflow is not already installed.

You can install MLflow from a terminal with:

```bash
pip install mlflow
```

Inside Jupyter you can run:

In [ ]:
# Uncomment if MLflow is not installed.
# %pip install mlflow

# 2. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from mlflow.models import infer_signature

# 3. Load the Advertising Dataset

The dataset used in this notebook contains 200 observations and four columns.

If `Advertising.csv` is in the same folder as this notebook, the following code will work directly.

In [ ]:
df = pd.read_csv("Advertising.csv")

df.head()

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Expected columns:

| Feature | Meaning |
|---|---|
| TV | Advertising expenditure on TV |
| Radio | Advertising expenditure on radio |
| Newspaper | Advertising expenditure on newspaper |
| Sales | Sales outcome |

We will use:

\[
X=[TV,\ Radio,\ Newspaper]
\]

and:

\[
y=Sales
\]

In [ ]:
X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]

X.head()

# 4. Train-Test Split

We use one fixed split so that every model is evaluated on the same data.

This makes model comparison fair.

In [ ]:
TEST_SIZE = 0.20
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

# 5. What Problem Does MLflow Solve?

Suppose we train many models:

```text
Linear Regression
Ridge
Lasso
Decision Tree
Random Forest
```

We may change:

- model type
- hyperparameters
- features
- train/test split
- preprocessing
- evaluation metrics

Soon we need answers to questions such as:

> Which model performed best?

> What hyperparameters produced that result?

> Where is that trained model?

> Can I reproduce that run?

> Which version is currently being used?

MLflow gives us a structured record of our ML experiments.

A useful mental model is:

\[
\boxed{\text{MLflow = memory for machine-learning experiments}}
\]

# 6. Core MLflow Vocabulary

The hierarchy is:

\[
\boxed{
\text{Experiment}
\rightarrow
\text{Run}
\rightarrow
\text{Parameters + Metrics + Artifacts}
}
\]

### Experiment

A collection of related ML runs.

Example:

```text
Advertising Model Comparison
```

### Run

One execution of the training process.

Example:

```text
Random Forest, max_depth=5
```

### Parameters

Things selected before or during model configuration.

Examples:

```text
max_depth = 5
n_estimators = 100
alpha = 1.0
test_size = 0.20
```

### Metrics

Numbers used to evaluate model performance.

Examples:

```text
RMSE
MAE
R²
Accuracy
Precision
Recall
```

### Artifacts

Files or objects generated by the run.

Examples:

```text
trained model
residual plot
feature importance plot
CSV
JSON
report
```

Remember:

\[
\boxed{\text{Parameters = choices}}
\]

\[
\boxed{\text{Metrics = measured results}}
\]

\[
\boxed{\text{Artifacts = files/objects}}
\]

# 7. Our First MLflow Experiment

We will start with Linear Regression.

First create or select an MLflow experiment.

In [ ]:
mlflow.set_experiment("Advertising Sales Prediction")

Now create one MLflow run.

Everything inside:

```python
with mlflow.start_run():
```

belongs to that run.

In [ ]:
with mlflow.start_run(run_name="Linear Regression - Manual Logging"):

    model = LinearRegression()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    # Parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", TEST_SIZE)
    mlflow.log_param("random_state", RANDOM_STATE)

    # Metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

    print("RMSE:", rmse)
    print("R2:", r2)

MLflow has now stored approximately:

```text
Advertising Sales Prediction
│
└── Linear Regression - Manual Logging
     │
     ├── Parameters
     │   ├── model_type
     │   ├── test_size
     │   └── random_state
     │
     └── Metrics
         ├── test_rmse
         └── test_r2
```

This is the first major transition:

Without MLflow:

\[
\text{Train} \rightarrow \text{See result}
\]

With MLflow:

\[
\text{Train} \rightarrow \text{Record the run}
\]

# 8. Opening the MLflow UI

After creating runs, open a terminal in the environment where MLflow is installed and run:

```bash
mlflow ui
```

A local server will normally start.

Open the address shown in the terminal, commonly:

```text
http://127.0.0.1:5000
```

In the UI you can inspect:

- experiments
- run names
- parameters
- metrics
- artifacts
- models

# 9. Comparing Multiple Models

Now we move from one run to a real experiment.

We will compare:

- Linear Regression
- Ridge
- Lasso
- Decision Tree
- Random Forest

Each model will become a **separate MLflow run**.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),

    "Ridge": Ridge(
        alpha=1.0
    ),

    "Lasso": Lasso(
        alpha=0.1
    ),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=5,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=5,
        random_state=42
    )
}

In [ ]:
mlflow.set_experiment("Advertising Model Comparison")

We will first do this with **manual logging** so the mechanics are completely clear.

In [ ]:
results = []

for model_name, model in models.items():

    with mlflow.start_run(run_name=model_name):

        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        train_rmse = mean_squared_error(y_train, train_pred) ** 0.5
        test_rmse = mean_squared_error(y_test, test_pred) ** 0.5
        test_mae = mean_absolute_error(y_test, test_pred)
        test_r2 = r2_score(y_test, test_pred)

        rmse_gap = test_rmse - train_rmse

        mlflow.log_param("model_name", model_name)
        mlflow.log_param("test_size", TEST_SIZE)
        mlflow.log_param("random_state", RANDOM_STATE)

        mlflow.log_metrics({
            "train_rmse": train_rmse,
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_r2": test_r2,
            "rmse_gap": rmse_gap
        })

        results.append({
            "model": model_name,
            "train_rmse": train_rmse,
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_r2": test_r2,
            "rmse_gap": rmse_gap
        })

results_df = pd.DataFrame(results)
results_df.sort_values("test_rmse")

## Why Log Train RMSE and Test RMSE?

Because model selection is not only about finding the largest \(R^2\).

We also care about generalization.

We define:

\[
\text{RMSE Gap}
=
\text{Test RMSE}
-
\text{Train RMSE}
\]

A very large gap may indicate overfitting.

Therefore model selection should consider:

\[
\boxed{
\text{Test performance}
+
\text{generalization}
+
\text{stability}
}
\]

This is an important MLOps mindset:

\[
\boxed{\text{Best training model} \neq \text{best production model}}
\]

# 10. Logging Artifacts

Metrics are only numbers.

MLflow can also store useful files associated with the training run.

For the Random Forest model, we will save:

1. the trained model
2. actual-vs-predicted plot
3. residual plot
4. feature importance plot

In [ ]:
mlflow.set_experiment("Advertising Artifacts Demo")

with mlflow.start_run(run_name="Random Forest with Artifacts"):

    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=5,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    mlflow.log_params({
        "n_estimators": 100,
        "max_depth": 5,
        "random_state": 42
    })

    mlflow.log_metrics({
        "test_rmse": rmse,
        "test_r2": r2
    })

    # Log the trained model
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="random_forest_model"
    )

    # Actual vs Predicted
    plt.figure(figsize=(7, 5))
    plt.scatter(y_test, y_pred)
    plt.xlabel("Actual Sales")
    plt.ylabel("Predicted Sales")
    plt.title("Actual vs Predicted Sales")
    plt.savefig("actual_vs_predicted.png", bbox_inches="tight")
    plt.close()

    mlflow.log_artifact("actual_vs_predicted.png")

    # Residual Plot
    residuals = y_test - y_pred

    plt.figure(figsize=(7, 5))
    plt.scatter(y_pred, residuals)
    plt.axhline(y=0)
    plt.xlabel("Predicted Sales")
    plt.ylabel("Residuals")
    plt.title("Residual Plot")
    plt.savefig("residual_plot.png", bbox_inches="tight")
    plt.close()

    mlflow.log_artifact("residual_plot.png")

    # Feature Importance
    plt.figure(figsize=(7, 5))
    plt.bar(X.columns, model.feature_importances_)
    plt.xlabel("Features")
    plt.ylabel("Importance")
    plt.title("Random Forest Feature Importance")
    plt.savefig("feature_importance.png", bbox_inches="tight")
    plt.close()

    mlflow.log_artifact("feature_importance.png")

    print("RMSE:", rmse)
    print("R2:", r2)

The run now conceptually contains:

```text
Random Forest with Artifacts
│
├── Parameters
├── Metrics
│
└── Artifacts
    ├── random_forest_model/
    ├── actual_vs_predicted.png
    ├── residual_plot.png
    └── feature_importance.png
```

Remember:

\[
\boxed{\text{Metric = number}}
\]

\[
\boxed{\text{Artifact = file or object}}
\]

# 11. MLflow Autologging

Manual logging is useful for learning, but it becomes repetitive.

MLflow provides autologging.

For Scikit-learn:

```python
mlflow.sklearn.autolog()
```

This allows MLflow to automatically capture much of the estimator information.

A very useful practical strategy is:

\[
\boxed{
\text{Autolog routine information}
+
\text{Manually log important project metrics}
}
\]

In [ ]:
# Turn on Scikit-learn autologging.
mlflow.sklearn.autolog()

In [ ]:
mlflow.set_experiment("Advertising Autologging")

with mlflow.start_run(run_name="Random Forest Autolog"):

    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=5,
        random_state=42
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_rmse = mean_squared_error(y_train, train_pred) ** 0.5
    test_rmse = mean_squared_error(y_test, test_pred) ** 0.5
    test_mae = mean_absolute_error(y_test, test_pred)
    test_r2 = r2_score(y_test, test_pred)

    rmse_gap = test_rmse - train_rmse

    # Custom project metrics
    mlflow.log_metrics({
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2,
        "rmse_gap": rmse_gap
    })

    print("Train RMSE:", train_rmse)
    print("Test RMSE:", test_rmse)
    print("Test MAE:", test_mae)
    print("Test R2:", test_r2)
    print("RMSE Gap:", rmse_gap)

## Autologging vs `start_run()`

Do not confuse these two ideas.

```python
mlflow.sklearn.autolog()
```

means:

> Automatically capture supported ML information.

Whereas:

```python
with mlflow.start_run():
```

means:

> Start one particular tracked run.

A simple analogy:

```text
autolog()    = turn the recorder on

start_run()  = begin one recording session
```

# 12. Multi-Model Comparison with Autologging

Now combine everything.

Each model gets:

- its own MLflow run
- automatic estimator logging
- manually logged project metrics

In [ ]:
mlflow.set_experiment("Advertising Model Comparison - Autolog")

autolog_results = []

for model_name, model in models.items():

    with mlflow.start_run(run_name=model_name):

        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        train_rmse = mean_squared_error(y_train, train_pred) ** 0.5
        test_rmse = mean_squared_error(y_test, test_pred) ** 0.5
        test_mae = mean_absolute_error(y_test, test_pred)
        test_r2 = r2_score(y_test, test_pred)

        rmse_gap = test_rmse - train_rmse

        mlflow.log_metrics({
            "train_rmse": train_rmse,
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_r2": test_r2,
            "rmse_gap": rmse_gap
        })

        mlflow.set_tag("model_name", model_name)

        autolog_results.append({
            "model": model_name,
            "train_rmse": train_rmse,
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_r2": test_r2,
            "rmse_gap": rmse_gap
        })

autolog_results_df = pd.DataFrame(autolog_results)

autolog_results_df.sort_values(
    "test_rmse"
)

# 13. Runs vs Registered Models

This distinction is fundamental.

## MLflow Run

A run is:

> One attempt at training or evaluating a model.

You may have hundreds or thousands of runs.

## Registered Model

A registered model is:

> A model deliberately promoted into the model-management system.

Typical workflow:

```text
TRAIN MANY
    ↓
TRACK ALL
    ↓
COMPARE
    ↓
SELECT
    ↓
REGISTER
```

Therefore:

\[
\boxed{\text{Tracking is broad; Registry is selective.}}
\]

# 14. Logging a Candidate Model with a Signature

A **model signature** documents the expected input and output schema.

For our Advertising model, the inputs are:

```text
TV
Radio
Newspaper
```

and the output is predicted Sales.

This becomes important during deployment because the model package carries information about the data format it expects.

In [ ]:
mlflow.set_experiment("Advertising Registry Demo")

with mlflow.start_run(run_name="Random Forest Candidate") as run:

    candidate_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=5,
        random_state=42
    )

    candidate_model.fit(X_train, y_train)

    candidate_pred = candidate_model.predict(X_test)

    candidate_rmse = mean_squared_error(
        y_test,
        candidate_pred
    ) ** 0.5

    candidate_r2 = r2_score(
        y_test,
        candidate_pred
    )

    mlflow.log_metrics({
        "test_rmse": candidate_rmse,
        "test_r2": candidate_r2
    })

    signature = infer_signature(
        X_train,
        candidate_model.predict(X_train)
    )

    mlflow.sklearn.log_model(
        sk_model=candidate_model,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(3)
    )

    run_id = run.info.run_id

print("Run ID:", run_id)

# 15. Registering the Model

Once the model has been logged inside a run, its URI can be expressed as:

```text
runs:/RUN_ID/model
```

We can create that URI:

In [ ]:
model_uri = f"runs:/{run_id}/model"

print(model_uri)

To register the candidate:

```python
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="Advertising_Sales_Model"
)
```

> **Important:** The exact registry setup depends on your MLflow tracking configuration.  
> Run the following cell when your registry is configured.

In [ ]:
# Uncomment when your MLflow Registry is configured.

# registered_model = mlflow.register_model(
#     model_uri=model_uri,
#     name="Advertising_Sales_Model"
# )

# registered_model

The structure then becomes:

```text
Advertising_Sales_Model
│
└── Version 1
```

If you later register another improved candidate using the same registered model name:

```text
Advertising_Sales_Model
│
├── Version 1
└── Version 2
```

The previous model is not simply overwritten.

This is **model versioning**.

# 16. Model Aliases

A useful production pattern is:

```text
champion
challenger
```

Example:

```text
Advertising_Sales_Model
│
├── Version 1
├── Version 2
├── Version 3   ← champion
└── Version 4   ← challenger
```

`champion` means:

> the currently preferred model.

`challenger` means:

> a new candidate being evaluated as a possible replacement.

The conceptual production pattern is:

\[
\boxed{\text{Champion vs Challenger}}
\]

A model should not necessarily become the production model just because it performs slightly better in one experiment.

# 17. Loading a Registered Model

A specific version can conceptually be loaded using a model URI such as:

```python
model = mlflow.sklearn.load_model(
    "models:/Advertising_Sales_Model/1"
)
```

An alias-based workflow is often cleaner:

```python
model = mlflow.sklearn.load_model(
    "models:/Advertising_Sales_Model@champion"
)
```

Why is an alias useful?

Because your application can keep asking for:

```text
Advertising_Sales_Model@champion
```

while the alias can be moved from one validated version to another.

Conceptually:

```text
Application
    │
    ▼
@champion
    │
    ▼
Model Version selected by Registry
```

# 18. Local Prediction from a Loaded Model

Suppose `model` is the loaded champion.

A new advertising scenario may look like this:

In [ ]:
new_data = pd.DataFrame({
    "TV": [150.0],
    "Radio": [25.0],
    "Newspaper": [30.0]
})

new_data

Then prediction would simply be:

```python
prediction = model.predict(new_data)
print(prediction)
```

If you have not configured the Registry yet, we can demonstrate with the locally trained candidate model:

In [ ]:
prediction = candidate_model.predict(new_data)

print("Predicted Sales:", prediction[0])

# 19. Loading vs Serving a Model

These are different ideas.

## Loading

```python
model = mlflow.sklearn.load_model(...)
```

means:

> Load the model inside a Python process.

## Serving

Serving means:

> Expose the model through an API so another application can send requests to it.

Conceptually:

```text
Website / App
      │
      ▼
HTTP Request
      │
      ▼
Model API
      │
      ▼
MLflow Model
      │
      ▼
Prediction
```

# 20. Serving an MLflow Model

A model can be served from the command line using MLflow's model-serving command.

A typical model URI may look like:

```text
models:/Advertising_Sales_Model@champion
```

Example command:

```bash
mlflow models serve \
-m "models:/Advertising_Sales_Model@champion" \
-p 5001
```

The exact serving options can depend on your MLflow version and environment.

A client application can then send data representing:

```json
{
  "dataframe_records": [
    {
      "TV": 150,
      "Radio": 25,
      "Newspaper": 30
    }
  ]
}
```

and receive a Sales prediction.

# 21. MLflow Architecture

This is where MLflow becomes an MLOps system rather than only a notebook utility.

At a high level:

```text
Notebook / Training Script
          │
          ▼
     MLflow Client
          │
          ▼
 MLflow Tracking Server
      /           \\
     /             \\
    ▼               ▼
Backend Store    Artifact Store
```

## Backend Store

Stores structured metadata such as:

- experiment information
- run IDs
- parameters
- metrics
- tags
- registry metadata

## Artifact Store

Stores files such as:

- trained models
- plots
- images
- reports
- other run artifacts

Therefore:

\[
\boxed{\text{Backend Store = metadata}}
\]

\[
\boxed{\text{Artifact Store = files and model artifacts}}
\]

# 22. Local MLflow vs Team MLflow

## Local Learning Setup

```text
Your Laptop
│
├── Notebook
├── MLflow
├── Tracking metadata
└── Artifacts
```

This is excellent for learning and individual experimentation.

## Team Setup

Imagine several data scientists.

Instead of each person having disconnected experiment histories:

```text
DS A ───┐
DS B ───┤
DS C ───┼────► Central MLflow Tracking Server
DS D ───┤
DS E ───┘
```

Everyone can use a shared tracking server.

The client can be pointed at a tracking server with:

```python
mlflow.set_tracking_uri(
    "http://your-mlflow-server:5000"
)
```

Then runs are sent to that shared MLflow environment.

# 23. Backend Store vs Artifact Store in Production

A production architecture may conceptually look like:

```text
Training Script
      │
      ▼
MLflow Tracking Server
      │
      ├────────────► Database
      │              experiments
      │              runs
      │              params
      │              metrics
      │              registry metadata
      │
      └────────────► Artifact / Object Storage
                     models
                     plots
                     artifacts
```

Why separate them?

Databases are excellent for structured metadata.

Artifact stores are more suitable for potentially large files such as:

```text
trained model binaries
large plots
reports
model packages
```

# 24. Model Lineage

Every MLflow run has a unique run ID.

That allows us to trace a production model back to the experiment that created it.

Conceptually:

```text
Production Model
      ↓
Model Version
      ↓
Source Run ID
      ↓
Parameters
      ↓
Metrics
      ↓
Artifacts
      ↓
Training Experiment
```

This is **lineage**.

It lets us answer:

> Which exact training run created the production model?

This is one of the most important MLOps benefits.

# 25. Reproducibility

MLflow helps reproducibility by keeping records of:

- model parameters
- metrics
- run IDs
- trained model artifacts
- model signatures
- environment-related model metadata
- source run information

However, complete reproducibility may also require external practices such as:

- Git for code versioning
- dataset versioning
- dependency/environment management
- fixed random seeds
- pipeline orchestration

So MLflow is an important part of reproducibility, but not the only part.

# 26. The Complete Advertising MLflow Workflow

```text
Advertising.csv
      │
      ▼
Prepare Data
      │
      ▼
Train Multiple Models
      │
      ├── Linear Regression
      ├── Ridge
      ├── Lasso
      ├── Decision Tree
      └── Random Forest
      │
      ▼
MLflow Tracking
      │
      ├── Parameters
      ├── Metrics
      ├── Tags
      ├── Models
      └── Artifacts
      │
      ▼
Compare Runs
      │
      ▼
Select Candidate
      │
      ▼
Model Registry
      │
      ▼
Versions
      │
      ▼
Champion / Challenger
      │
      ▼
Load or Serve Model
      │
      ▼
Application Prediction
```

# 27. The Most Important Ideas to Remember

### 1. MLflow gives memory to ML experiments

\[
\boxed{\text{What did I try, how did I try it, and what happened?}}
\]

### 2. Experiment and Run are different

```text
Experiment
├── Run 1
├── Run 2
└── Run 3
```

### 3. Parameters, metrics, and artifacts are different

\[
\boxed{\text{Parameters = choices}}
\]

\[
\boxed{\text{Metrics = measured numbers}}
\]

\[
\boxed{\text{Artifacts = files and model objects}}
\]

### 4. Autologging reduces repetitive tracking code

\[
\boxed{
\text{Autolog routine data}
+
\text{log project-specific metrics manually}
}
\]

### 5. Tracking and Registry are different

\[
\boxed{\text{Runs = everything we tried}}
\]

\[
\boxed{\text{Registry = models deliberately selected for management}}
\]

### 6. Versioning avoids the "final_model_v7.pkl" problem

Instead of:

```text
model_final.pkl
model_final2.pkl
model_really_final.pkl
```

we can manage:

```text
Advertising_Sales_Model
Version 1
Version 2
Version 3
```

### 7. Production model selection is more than highest test score

Consider:

- test performance
- train/test gap
- stability
- validation
- operational suitability

### 8. Registry aliases decouple applications from model version numbers

```text
Application
    ↓
@champion
    ↓
current validated model version
```

### 9. Lineage gives traceability

\[
\boxed{
\text{Production Model}
\rightarrow
\text{Version}
\rightarrow
\text{Run}
\rightarrow
\text{Parameters + Metrics + Artifacts}
}
\]

# 28. Practice Questions

Try answering these without looking back.

1. What is an MLflow experiment?
2. What is an MLflow run?
3. What is the difference between a parameter and a metric?
4. Is a residual plot a metric or an artifact?
5. Why might we log both train RMSE and test RMSE?
6. What does the RMSE gap tell us?
7. Why should every experiment not automatically become a registered model?
8. What is model versioning?
9. What is the purpose of a `champion` alias?
10. What is the difference between loading and serving a model?
11. What does the backend store contain?
12. What does the artifact store contain?
13. What is model lineage?
14. Why is MLflow useful for a team rather than only one data scientist?
15. Why is the best training model not necessarily the best production model?

# 29. Hands-On Exercise

Modify the experiment by adding:

```python
GradientBoostingRegressor
```

Then:

1. train it on the same training data,
2. track it as a separate MLflow run,
3. log train RMSE,
4. log test RMSE,
5. log MAE,
6. log \(R^2\),
7. calculate the RMSE gap,
8. compare it with Random Forest,
9. decide which model you would nominate as the candidate,
10. explain your decision.

This is the transition from simply learning MLflow syntax to thinking like an MLOps practitioner.

# 30. Final Mental Model

If you remember only one diagram, remember this:

```text
Notebook / Training Script
          ↓
       MLflow Run
          ↓
Parameters + Metrics + Artifacts
          ↓
     Compare Runs
          ↓
    Select Candidate
          ↓
     Model Registry
          ↓
 Versions + Aliases
          ↓
     Load / Serve
          ↓
      Production
          ↓
        Monitor
          ↓
        Retrain
```

And the simplest summary is:

\[
\boxed{
\text{MLflow tracks experiments, manages model versions, and connects model development to deployment.}
}
\]